In [2]:
import json
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd

def extract_pattern_pairs(log_path: str) -> List[Dict[str, Any]]:
    """Parse a summarization log and return pattern name collections per cluster."""
    lines = Path(log_path).read_text(encoding="utf-8").splitlines()

    entries: List[Dict[str, Any]] = []
    cluster_id: int | None = None
    section: str | None = None
    buffer: List[str] = []
    original_names: List[str] = []
    summarized_names: List[str] = []
    thinkings: str = ""

    def flush(active_section: str | None) -> None:
        nonlocal buffer, original_names, summarized_names, thinkings
        if not active_section:
            buffer = []
            return
        raw = "\n".join(buffer).strip()
        buffer = []
        if not raw:
            return
        if active_section == "original":
            try:
                data = json.loads(raw)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Failed to parse original patterns for cluster {cluster_id}") from exc
            original_names = [item.get("Pattern Name") for item in data if isinstance(item, dict)]
        elif active_section == "summary":
            try:
                data = json.loads(raw)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Failed to parse summarized patterns for cluster {cluster_id}") from exc
            summarized_names = [
                item.get("Pattern Name")
                for item in data.get("patterns", [])
                if isinstance(item, dict)
            ]
            thinkings = data.get("thinkings","")

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("Cluster ") and stripped.split()[1].isdigit():
            flush(section)
            if cluster_id is not None:
                entries.append({
                    "cluster": cluster_id,
                    "original_patterns": original_names,
                    "summarized_patterns": summarized_names,
                    "thinking": thinkings,
                })
            cluster_id = int(stripped.split()[1])
            section = None
            original_names = []
            summarized_names = []
        elif stripped == "Original Patterns:":
            flush(section)
            section = "original"
        elif stripped == "Summarized Patterns:":
            flush(section)
            section = "summary"
        elif stripped == "Verification Result:":
            flush(section)
            section = None
        elif section:
            buffer.append(line)

    flush(section)
    if cluster_id is not None:
        entries.append({
            "cluster": cluster_id,
            "original_patterns": original_names,
            "summarized_patterns": summarized_names,
            "thinking": thinkings,
        })

    return entries

In [48]:
import pandas as pd,ast

pattern_pairs_df_01 = pd.read_csv("./summarized_patterns/raw/iter_01.csv")
pattern_pairs_df_01.to_csv("./summarized_patterns/raw/iter_01.csv")
iter_01_summarized_patterns = set(pattern_pairs_df_01["summarized_patterns"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x).explode())

pattern_pairs_df_02 = pd.read_csv("./summarized_patterns/raw/iter_02.csv")
pattern_pairs_df_02.to_csv("./summarized_patterns/raw/iter_02.csv")
iter_02_summarized_patterns = set(pattern_pairs_df_02["summarized_patterns"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x).explode())

pattern_pairs_df_03 = pd.read_csv("./summarized_patterns/raw/iter_03.csv")
pattern_pairs_df_03.to_csv("./summarized_patterns/raw/iter_03.csv")
iter_03_summarized_patterns = set(pattern_pairs_df_03["summarized_patterns"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x).explode())

intersection_patterns_01 = iter_01_summarized_patterns & iter_02_summarized_patterns
print(f"Number of patterns common in 01 & 02 iterations: {len(intersection_patterns_01)}")
print("Common patterns:")
for pattern in intersection_patterns_01:
    print(f" - {pattern}")

intersection_patterns_02 = iter_02_summarized_patterns & iter_03_summarized_patterns
print(f"\n\nNumber of patterns common in 02 & 03 iterations: {len(intersection_patterns_02)}")
print("Common patterns:")
for pattern in intersection_patterns_02:
    print(f" - {pattern}")

intersection_patterns = iter_01_summarized_patterns & iter_02_summarized_patterns & iter_03_summarized_patterns
print(f"\n\nNumber of patterns common in all 3 iterations: {len(intersection_patterns)}")
print("Common patterns:")
for pattern in intersection_patterns:
    print(f" - {pattern}")

Number of patterns common in 01 & 02 iterations: 26
Common patterns:
 - Proactive Static Planning
 - Nearest Neighbor Output Augmentation
 - Denoising Pretraining for Foundational Generative Models
 - Autonomous Tool Generation
 - Efficient Dense Semantic Retrieval
 - Adversarial Robustness Evaluation
 - Persona and Contextual Framing
 - Contextual Refinement
 - LLM-Guided Human Prompt Refinement
 - Structured Output Generation
 - Reasoning-Action Alignment
 - Progressive Response Disclosure
 - Parameter-Efficient LLM Adaptation
 - Retrieval-Augmented Generation (RAG)
 - User Intent Resolution
 - Explicit Step-by-Step Reasoning (Chain-of-Thought)
 - Ambiguity-Robust Demonstrations
 - RAG KV Cache Optimization System
 - Task Conditioning with Control Tokens
 - AI System Process Transparency and Trust Calibration
 - Augmented Response Synthesis
 - In-Prompt Guardrails
 - LLM Fallback to Inherent Knowledge
 - Direct LLM Generation
 - Personalized Tool Interaction
 - Exploratory Reasoning 

In [ ]:
import ast

pdf1 = pd.read_csv("data/patterns-data/iter_01.csv")
pdf1_set = set(pdf1["summarized_patterns"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x).explode())
print(pdf1.head())
print(len(pdf1_set))

pdf2 = pd.read_csv("data/patterns-data/iter_02.csv")
pdf2_set = set(pdf2["summarized_patterns"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x).explode())
print(len(pdf2_set))

pdf3 = pd.read_csv("data/patterns-data/iter_03.csv")
pdf3_set = set(pdf3["summarized_patterns"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x).explode())
print(len(pdf3_set))

#patterns_common_to_all_iters = pdf1_set.intersection(pdf2_set).intersection(pdf3_set)
#print(len(patterns_common_to_all_iters))

print("p1_p2_common")
p1_p2_common = pdf1_set & pdf2_set
print(p1_p2_common)


In [ ]:
import ast
pattern_pairs_df_03["summarized_patterns"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x).explode()

0             [LLM-Tool Orchestration and Augmentation]
1     [AI Tooling and Capability Augmentation, Synth...
2            [Robustness and Safe External Interaction]
3                    [Intelligent Input Interpretation]
4                       [Personalized Tool Interaction]
5                  [LLM Fallback to Inherent Knowledge]
6     [Responsible AI Design: Explainability, Fairne...
7           [LLM Uncertainty Management and Abstention]
8     [Calibrated Transparency and Progressive Discl...
9                [Prompt Engineering for Model Control]
10                     [Persona and Contextual Framing]
11              [Task Conditioning with Control Tokens]
12    [Adaptive Execution Cycle, Reasoning-Action Al...
13    [Structured Reasoning, Hierarchical Problem So...
14                       [Automated Prompt Engineering]
15                       [Dynamic Contextual Prompting]
16                 [LLM-Guided Human Prompt Refinement]
17               [Retrieval-Augmented Generation

In [41]:
log_file = "./logs/summarization_log_iter_01.txt"
pattern_pairs_01 = extract_pattern_pairs(log_file)
pattern_pairs_df_01 = pd.DataFrame(pattern_pairs_01)
pattern_pairs_df_01.to_csv("./summarized_patterns/raw/iter_01.csv")
iter_01_summarized_patterns = set(pattern_pairs_df_01["summarized_patterns"].explode())

In [42]:
log_file = "./logs/summarization_log_iter_02.txt"
pattern_pairs_02 = extract_pattern_pairs(log_file)
pattern_pairs_df_02 = pd.DataFrame(pattern_pairs_02)
pattern_pairs_df_02.to_csv("./summarized_patterns/raw/iter_02.csv")
iter_02_summarized_patterns = set(pattern_pairs_df_02["summarized_patterns"].explode())

In [43]:
log_file = "./logs/summarization_log_iter_03.txt"
pattern_pairs_03 = extract_pattern_pairs(log_file)
pattern_pairs_df_03 = pd.DataFrame(pattern_pairs_03)
pattern_pairs_df_03.to_csv("./summarized_patterns/raw/iter_03.csv")
iter_03_summarized_patterns = set(pattern_pairs_df_03["summarized_patterns"].explode())

In [40]:
intercection_patterns = iter_01_summarized_patterns & iter_02_summarized_patterns
print(f"Number of patterns common in 01 & 02 iterations: {len(intercection_patterns)}")
print("Common patterns:")
for pattern in intercection_patterns:
    print(f" - {pattern}")

intercection_patterns = iter_02_summarized_patterns & iter_03_summarized_patterns
print(f"\n\nNumber of patterns common in 02 & 03 iterations: {len(intercection_patterns)}")
print("Common patterns:")
for pattern in intercection_patterns:
    print(f" - {pattern}")

intercection_patterns = iter_01_summarized_patterns & iter_02_summarized_patterns & iter_03_summarized_patterns
print(f"\n\nNumber of patterns common in all 3 iterations: {len(intercection_patterns)}")
print("Common patterns:")
for pattern in intercection_patterns:
    print(f" - {pattern}")

Number of patterns common in 01 & 02 iterations: 26
Common patterns:
 - Proactive Static Planning
 - Nearest Neighbor Output Augmentation
 - Denoising Pretraining for Foundational Generative Models
 - Autonomous Tool Generation
 - Efficient Dense Semantic Retrieval
 - Adversarial Robustness Evaluation
 - Persona and Contextual Framing
 - Contextual Refinement
 - LLM-Guided Human Prompt Refinement
 - Structured Output Generation
 - Reasoning-Action Alignment
 - Progressive Response Disclosure
 - Parameter-Efficient LLM Adaptation
 - Retrieval-Augmented Generation (RAG)
 - User Intent Resolution
 - Explicit Step-by-Step Reasoning (Chain-of-Thought)
 - Ambiguity-Robust Demonstrations
 - RAG KV Cache Optimization System
 - Task Conditioning with Control Tokens
 - AI System Process Transparency and Trust Calibration
 - Augmented Response Synthesis
 - In-Prompt Guardrails
 - LLM Fallback to Inherent Knowledge
 - Direct LLM Generation
 - Personalized Tool Interaction
 - Exploratory Reasoning 

In [34]:
pattern_pairs_df_01["original_patterns"] = pattern_pairs_df_01["original_patterns"].apply(lambda x: str([f"{i+1}. {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))
pattern_pairs_df_02["original_patterns"] = pattern_pairs_df_02["original_patterns"].apply(lambda x: str([f"{i+1}. {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))
pattern_pairs_df_03["original_patterns"] = pattern_pairs_df_03["original_patterns"].apply(lambda x: str([f"{i+1}. {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))

pattern_pairs_df_01["summarized_patterns"] = pattern_pairs_df_01["summarized_patterns"].apply(lambda x: str([f"{i+1}. {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))
pattern_pairs_df_02["summarized_patterns"] = pattern_pairs_df_02["summarized_patterns"].apply(lambda x: str([f"{i+1}. {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))
pattern_pairs_df_03["summarized_patterns"] = pattern_pairs_df_03["summarized_patterns"].apply(lambda x: str([f"{i+1}. {row}" for i,row in enumerate(x)]).replace("[","").replace("]","").replace(",","\n").replace("'",""))

In [35]:
pattern_pairs_df_01[["original_patterns","summarized_patterns","thinking"]].to_csv("./summarized_patterns/comparison/iter_01_patterns.csv", index=False)
pattern_pairs_df_02[["original_patterns","summarized_patterns","thinking"]].to_csv("./summarized_patterns/comparison/iter_02_patterns.csv", index=False)
pattern_pairs_df_03[["original_patterns","summarized_patterns","thinking"]].to_csv("./summarized_patterns/comparison/iter_03_patterns.csv", index=False)

In [36]:
for col in ["original_patterns","summarized_patterns","thinking"]:
    pattern_pairs_df_01[col] = pattern_pairs_df_01[col].apply(lambda x: str(x).replace("\n","<br>"))
    pattern_pairs_df_02[col] = pattern_pairs_df_02[col].apply(lambda x: str(x).replace("\n","<br>"))
    pattern_pairs_df_03[col] = pattern_pairs_df_03[col].apply(lambda x: str(x).replace("\n","<br>"))

pattern_pairs_df_01[["original_patterns","summarized_patterns"]].to_markdown("./summarized_patterns/comparison/iter_01_patterns.md")
pattern_pairs_df_02[["original_patterns","summarized_patterns"]].to_markdown("./summarized_patterns/comparison/iter_02_patterns.md")
pattern_pairs_df_03[["original_patterns","summarized_patterns"]].to_markdown("./summarized_patterns/comparison/iter_03_patterns.md")